# Cross-table features

Signals that combine multiple source tables — the only home for cross-table logic. Two layers: hand-picked **domain** features (exposure, burden, history, delinquency) and a **programmatic sweep** of every monetary aggregate against income / current-loan anchors. Column names are discovered at runtime, so it adapts to whatever the feature notebooks produced. Internal only (no `EXT_SOURCE`), prefixed `x_`; selection prunes the rest.

In [8]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from src.data import load_master, save_features

X, y = load_master(all_rows=True)
internal = [c for c in X.columns if not (c.lower().startswith(("ext_", "x_")) or "ext_source" in c.lower())]
C = set(internal)

def col(n):
    return X[n] if n in C else pd.Series(np.nan, index=X.index)
def ratio(a, b):
    return (col(a) / col(b)).replace([np.inf, -np.inf], np.nan)
def total(cols):
    return X[cols].fillna(0).sum(axis=1) if cols else pd.Series(np.nan, index=X.index)

cross = pd.DataFrame(index=X.index)
len(internal)

3021

## Iteration 3 — domain features

In [9]:
# exposure / leverage (debt & credit vs income and current loan)
cross["x_bureau_debt_to_income"]    = ratio("bureau_AMT_CREDIT_SUM_DEBT_sum", "AMT_INCOME_TOTAL")
cross["x_bureau_credit_to_income"]  = ratio("bureau_AMT_CREDIT_SUM_sum", "AMT_INCOME_TOTAL")
cross["x_bureau_credit_to_current"] = ratio("bureau_AMT_CREDIT_SUM_sum", "AMT_CREDIT")
cross["x_bureau_debt_to_current"]   = ratio("bureau_AMT_CREDIT_SUM_DEBT_sum", "AMT_CREDIT")
cross["x_cc_balance_to_income"]     = ratio("cc_AMT_BALANCE_mean", "AMT_INCOME_TOTAL")

# payment burden (all annuities vs income)
cross["x_total_annuity_to_income"]  = (col("AMT_ANNUITY").fillna(0) + col("bureau_AMT_ANNUITY_sum").fillna(0)) / col("AMT_INCOME_TOTAL")
cross["x_prev_annuity_to_income"]   = ratio("prev_AMT_ANNUITY_sum", "AMT_INCOME_TOTAL")

# current loan vs their history
cross["x_credit_to_prev_credit"]    = ratio("AMT_CREDIT", "prev_AMT_CREDIT_mean")
cross["x_credit_to_bureau_credit"]  = ratio("AMT_CREDIT", "bureau_AMT_CREDIT_SUM_mean")
cross["x_annuity_to_prev_annuity"]  = ratio("AMT_ANNUITY", "prev_AMT_ANNUITY_mean")
cross["x_goods_to_prev_goods"]      = ratio("AMT_GOODS_PRICE", "prev_AMT_GOODS_PRICE_mean")

# breadth of credit
cross["x_total_prior_loans"]        = col("bureau_count").fillna(0) + col("prev_count").fillna(0)
cross["x_prev_to_bureau_loans"]     = ratio("prev_count", "bureau_count")
cross["x_bureau_overdue_to_debt"]   = ratio("bureau_AMT_CREDIT_SUM_OVERDUE_sum", "bureau_AMT_CREDIT_SUM_DEBT_sum")
cross.shape

(356255, 14)

## Cross-table totals & worst-of

In [10]:
debt_cols  = [c for c in internal if "DEBT" in c and c.endswith("_sum")] + \
             [c for c in internal if c.startswith("cc") and "BALANCE" in c and c.endswith("_mean")]
count_cols = [c for c in internal if c.endswith("_count")]
dpd_cols   = [c for c in internal if ("DPD" in c or "days_late" in c) and c.endswith("_max")]

cross["x_total_debt"]           = total(debt_cols)
cross["x_total_debt_to_income"] = cross["x_total_debt"] / col("AMT_INCOME_TOTAL")
cross["x_total_credit_lines"]   = total(count_cols)
cross["x_worst_dpd_any"]        = X[dpd_cols].max(axis=1) if dpd_cols else np.nan
cross.shape

(356255, 18)

## Programmatic sweep — monetary aggregates vs anchors

In [11]:
anchors = {"income": "AMT_INCOME_TOTAL", "credit": "AMT_CREDIT", "annuity": "AMT_ANNUITY"}
money = [c for c in internal if "AMT" in c and c.endswith(("_sum", "_mean")) and not c.startswith("AMT")]
for c in money:
    for name, a in anchors.items():
        cross[f"x_{c}_to_{name}"] = ratio(c, a)
print(f"swept {len(money)} monetary aggregates x {len(anchors)} anchors")
cross.shape

swept 142 monetary aggregates x 3 anchors


(356255, 444)

## Iteration 4 — cross-source risk, capacity & velocity

In [12]:
# worst / average delinquency across every history source (bureau, ins, pos, cc)
late_cols = [c for c in internal if any(k in c.lower() for k in ("late_share", "del_share", "dpd_share"))]
cross["x_max_late_share_any"]  = X[late_cols].max(axis=1) if late_cols else np.nan
cross["x_mean_late_share_any"] = X[late_cols].mean(axis=1) if late_cols else np.nan

# total overdue across the bureau, relative to income
overdue_cols = [c for c in internal if "OVERDUE" in c and c.endswith("_sum")]
cross["x_total_overdue"]           = total(overdue_cols)
cross["x_total_overdue_to_income"] = cross["x_total_overdue"] / col("AMT_INCOME_TOTAL")

# repayment capacity vs ALL obligations (current + bureau + prev annuities)
tot_annuity = col("AMT_ANNUITY").fillna(0) + col("bureau_AMT_ANNUITY_sum").fillna(0) + col("prev_AMT_ANNUITY_sum").fillna(0)
cross["x_all_annuity_to_income"]        = tot_annuity / col("AMT_INCOME_TOTAL")
cross["x_disposable_after_all_annuity"] = col("AMT_INCOME_TOTAL") - tot_annuity
cross["x_income_to_total_obligations"]  = col("AMT_INCOME_TOTAL") / (cross["x_total_debt"] + col("AMT_CREDIT").fillna(0))

# active vs closed exposure (bureau iteration-2 splits), refusal history (prev splits)
cross["x_active_debt_to_income"] = ratio("bureau_active_AMT_CREDIT_SUM_DEBT_sum", "AMT_INCOME_TOTAL")
cross["x_closed_debt_to_income"] = ratio("bureau_closed_AMT_CREDIT_SUM_DEBT_sum", "AMT_INCOME_TOTAL")
cross["x_refused_to_approved"]   = ratio("prev_ref_count", "prev_appr_count")
cross["x_approved_credit_share"] = ratio("prev_appr_AMT_CREDIT_sum", "prev_AMT_CREDIT_sum")

# household-normalized burden
cross["x_total_debt_per_person"] = cross["x_total_debt"] / col("CNT_FAM_MEMBERS")
cross["x_all_obligation_per_person"] = (col("AMT_CREDIT").fillna(0) + cross["x_total_debt"]) / col("CNT_FAM_MEMBERS")
cross = cross.replace([np.inf, -np.inf], np.nan)
cross.shape

(356255, 457)

## Coverage

In [13]:
allnan = [c for c in cross.columns if cross[c].isna().all()]
print(f"{cross.shape[1]} cross features | {len(allnan)} all-NaN (unresolved names): {allnan[:8]}")

457 cross features | 0 all-NaN (unresolved names): []


# Save

In [14]:
save_features(cross, "cross"); cross.shape

(356255, 457)